# Paper-aligned No-Context baseline trên Stanford Cars

Notebook này là bản copy/adapt từ `baseline_no_tacs_image_classification.ipynb`, nhưng phần dataset đã đổi từ CUB-200-2011 sang Stanford Cars trong thư mục `archive (10)`.

Pipeline giữ nguyên tinh thần baseline No-Context: `query image -> DINOv3 ViT-S/16 -> classification head -> class_id 0..195 + tên dòng xe`.

Theo yêu cầu bám **standard split của dataset**, notebook chỉ dùng hai split có sẵn: train và test. Toàn bộ `cars_train_annos.mat` được dùng để train; `cars_test_annos.mat` được dùng làm official test. Lưu ý: trong `archive (10)`, test annotations không có trường `class`, nên notebook không thể tính test accuracy nếu không có thêm file nhãn test.

## 1. Dataset Stanford Cars trong `archive (10)`

Cấu trúc dataset đã đọc được:

```text
archive (10)/
  car_devkit/devkit/
    cars_meta.mat          # 196 tên lớp xe
    cars_train_annos.mat   # 8,144 ảnh train có bbox + class + fname
    cars_test_annos.mat    # 8,041 ảnh test có bbox + fname, không có class trong bản này
  cars_train/cars_train/
    00001.jpg ...
  cars_test/cars_test/
    00001.jpg ...
```

Class ID trong file `.mat` là `1..196`; code chuyển về nhãn mô hình `0..195`. Không tạo validation split tự phát sinh, vì đề yêu cầu dùng standard train/test split của dataset.

In [ ]:
%pip -q install torch torchvision scipy matplotlib pandas pillow tqdm

import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from scipy.io import loadmat
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

## 2. Cấu hình thí nghiệm

Notebook vẫn dùng backbone `dinov3_vits16` như bản baseline trước. Phần dataset chuyển sang Stanford Cars `.mat` annotations, `NUM_CLASSES` là 196, và output folder là `outputs/baseline_no_tacs_cars`.

`DATASET_ROOT` được dò theo nhiều vị trí để chạy được cả trong workspace local và Colab/Drive nếu bạn copy dataset sang đó.

In [ ]:
PROJECT_DIR = Path.cwd()
DATASET_ROOT_CANDIDATES = [
    PROJECT_DIR / 'archive (10)',
    Path(r'F:\Documents\CODE\Python\cv_project\XLA\archive (10)'),
    Path('/content/archive (10)'),
]
DATASET_ROOT = next(
    (
        path for path in DATASET_ROOT_CANDIDATES
        if (path / 'car_devkit' / 'devkit' / 'cars_meta.mat').is_file()
    ),
    DATASET_ROOT_CANDIDATES[0],
)
DEVKIT_ROOT = DATASET_ROOT / 'car_devkit' / 'devkit'
TRAIN_IMAGE_ROOT = DATASET_ROOT / 'cars_train' / 'cars_train'
TEST_IMAGE_ROOT = DATASET_ROOT / 'cars_test' / 'cars_test'
PREDICT_IMAGE = TRAIN_IMAGE_ROOT / '00001.jpg'
OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'baseline_no_tacs_cars'
OUTPUT_DIR = OUT_PUT_DIR  # alias de cac cell cu van dung duoc
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# DINOv3 ViT-S/16 configuration.
DINOV3_REPO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINOV3_REPO_DIR = str(PROJECT_DIR / 'dinov3')
DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS_URL = 'https://dl.fbaipublicfiles.com/dinov3/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
DINOV3_WEIGHTS = str(PROJECT_DIR / 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth')
BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
NUM_CLASSES = 196
TOP_K = 5
USE_BBOX_CROP = False

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('TRAIN_IMAGE_ROOT:', TRAIN_IMAGE_ROOT.resolve())
print('TEST_IMAGE_ROOT:', TEST_IMAGE_ROOT.resolve())
print('DEVKIT_ROOT:', DEVKIT_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Config: backbone={DINOV3_MODEL_NAME}, image={IMAGE_SIZE}, batch={BATCH_SIZE}, epochs={EPOCHS}, classes={NUM_CLASSES}')


## 3. Transform và standard train/test split

Stanford Cars bản này đã có train/test split riêng. Notebook giữ đúng split đó:

- Train: `cars_train_annos.mat` + ảnh trong `cars_train/cars_train`.
- Test: `cars_test_annos.mat` + ảnh trong `cars_test/cars_test`.

Do test annotations không có nhãn trong `archive (10)`, test loader sẽ trả về `(image, fname)` để xuất file dự đoán. Không tách validation từ train.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.75, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

def _mat_array_to_list(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, list):
        return value
    return [value]

def load_cars_class_names(devkit_root):
    metadata = loadmat(devkit_root / 'cars_meta.mat', squeeze_me=True, struct_as_record=False)
    class_names = _mat_array_to_list(metadata['class_names'])
    return [str(name) for name in class_names]

def load_cars_annotations(devkit_root, split):
    if split not in {'train', 'test'}:
        raise ValueError("split phải là 'train' hoặc 'test'")
    annotation_file = devkit_root / f'cars_{split}_annos.mat'
    raw = loadmat(annotation_file, squeeze_me=True, struct_as_record=False)
    annotations = _mat_array_to_list(raw['annotations'])
    rows = []
    for annotation in annotations:
        row = {
            'fname': str(getattr(annotation, 'fname')),
            'bbox_x1': int(getattr(annotation, 'bbox_x1')),
            'bbox_y1': int(getattr(annotation, 'bbox_y1')),
            'bbox_x2': int(getattr(annotation, 'bbox_x2')),
            'bbox_y2': int(getattr(annotation, 'bbox_y2')),
        }
        if hasattr(annotation, 'class'):
            row['class_id'] = int(getattr(annotation, 'class')) - 1
        rows.append(row)
    return pd.DataFrame(rows)

class StanfordCarsDataset(Dataset):
    def __init__(self, annotations, image_root, transform, class_names=None, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform
        self.class_names = class_names
        self.crop_bbox = crop_bbox
        self.has_labels = 'class_id' in self.annotations.columns

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = Image.open(self.image_root / row['fname']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = int(row['bbox_x2'])
            bottom = int(row['bbox_y2'])
            image = image.crop((left, top, right, bottom))
        image = self.transform(image)
        if not self.has_labels:
            return image, row['fname']
        return image, int(row['class_id'])

class_names = load_cars_class_names(DEVKIT_ROOT)
NUM_CLASSES = len(class_names)
train_annotations = load_cars_annotations(DEVKIT_ROOT, 'train')
test_annotations = load_cars_annotations(DEVKIT_ROOT, 'test')

missing_train_images = [fname for fname in train_annotations['fname'] if not (TRAIN_IMAGE_ROOT / fname).is_file()]
missing_test_images = [fname for fname in test_annotations['fname'] if not (TEST_IMAGE_ROOT / fname).is_file()]
if missing_train_images:
    raise FileNotFoundError(f'Thiếu ảnh train, ví dụ: {missing_train_images[:5]}')
if missing_test_images:
    raise FileNotFoundError(f'Thiếu ảnh test, ví dụ: {missing_test_images[:5]}')

train_df = train_annotations.reset_index(drop=True)
test_df = test_annotations.reset_index(drop=True)

train_dataset = StanfordCarsDataset(train_df, TRAIN_IMAGE_ROOT, train_transform, class_names=class_names, crop_bbox=USE_BBOX_CROP)
test_dataset = StanfordCarsDataset(test_df, TEST_IMAGE_ROOT, eval_transform, class_names=class_names, crop_bbox=USE_BBOX_CROP)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print('Classes:', NUM_CLASSES)
print(f'Standard train images: {len(train_dataset):,}')
print(f'Standard test images: {len(test_dataset):,}')
print(f'Train label range: {int(train_annotations.class_id.min())}..{int(train_annotations.class_id.max())}')
print('Test annotation has class label:', 'class_id' in test_annotations.columns)
print('Nếu dòng trên là False, notebook sẽ chỉ xuất prediction cho test, không tính accuracy.')
display(train_annotations.head())
display(test_annotations.head())

In [ ]:
class_counts = train_df['class_id'].value_counts().sort_index()
class_count_df = pd.DataFrame({
    'class_id': class_counts.index,
    'car_name': [class_names[i] for i in class_counts.index],
    'count': class_counts.values,
})
display(class_count_df)

sample_images, sample_labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for image_tensor, label, ax in zip(sample_images[:8], sample_labels[:8], axes.flat):
    image = image_tensor.permute(1, 2, 0).numpy() * np.asarray(imagenet_std) + np.asarray(imagenet_mean)
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(f'{int(label)}: {class_names[int(label)]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Kiểm tra hoặc tải DINOv3 assets

Cell này chuẩn bị repo DINOv3 và checkpoint ViT-S/16. Nếu máy đã có file weight, hãy đặt `DINOV3_WEIGHTS` trong cell cấu hình trỏ tới file đó. Nếu chưa có, cell này sẽ thử clone repo DINOv3 và tải checkpoint về workspace hiện tại.

In [ ]:
def ensure_dinov3_assets():
    repo_dir = Path(DINOV3_REPO_DIR)
    weights_path = Path(DINOV3_WEIGHTS)

    if not repo_dir.is_dir():
        try:
            print('Cloning DINOv3 repo to:', repo_dir)
            subprocess.run(['git', 'clone', '--depth', '1', DINOV3_REPO_URL, str(repo_dir)], check=True)
        except Exception as exc:
            print('Không clone được DINOv3 repo. torch.hub sẽ thử tải từ GitHub khi tạo model.')
            print(type(exc).__name__ + ':', exc)
    else:
        print('DINOv3 repo đã tồn tại:', repo_dir)

    if not weights_path.is_file():
        print('Đang tải DINOv3 ViT-S/16 weights về:', weights_path)
        urlretrieve(DINOV3_WEIGHTS_URL, weights_path)
    else:
        print('DINOv3 weights đã tồn tại:', weights_path)

    if weights_path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError(f'File weight có vẻ bị lỗi hoặc quá nhỏ: {weights_path}')

    print('Weights size MB:', round(weights_path.stat().st_size / 1024**2, 2))

ensure_dinov3_assets()

## 5. Mô hình baseline ViT không TACS

Baseline dùng backbone `dinov3_vits16`, lấy CLS embedding 384 chiều từ DINOv3 ViT-S/16 rồi thêm classification head 196 lớp cho Stanford Cars. Toàn bộ backbone và head được fine-tune bằng AdamW; không có Selector, retrieval hoặc context fusion.

In [ ]:
class DinoV3NoContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights, num_classes, feature_dim=384):
        super().__init__()
        repo_dir = Path(repo_dir)
        source = 'local' if repo_dir.is_dir() else 'github'
        repo_or_dir = str(repo_dir) if repo_dir.is_dir() else 'facebookresearch/dinov3'
        self.backbone = torch.hub.load(repo_or_dir, model_name, source=source, weights=weights)
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, images):
        features = self.backbone(images)
        if isinstance(features, dict):
            features = features['x_norm_clstoken']
        return self.head(features)

def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)

    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    if path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError("File weight bị rỗng hoặc bị hỏng")

    return str(path)

def create_paper_vit_s16():
    network = DinoV3NoContextClassifier(
        repo_dir=DINOV3_REPO_DIR,
        model_name=DINOV3_MODEL_NAME,
        weights=resolve_dinov3_weights(),
        num_classes=NUM_CLASSES,
        feature_dim=FEATURE_DIM,
    )
    print(f'Loaded DINOv3 backbone: {DINOV3_MODEL_NAME}')
    print(f'DINOv3 weights: {resolve_dinov3_weights()}')
    return network

model = create_paper_vit_s16().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
checkpoint_path = OUT_PUT_DIR / 'final_no_context_dinov3_vits16_cars.pt'
last_checkpoint_path = OUT_PUT_DIR / 'checkpoint_last.pt'
print(model)


## 6. Training

Training chi dung standard train split cua dataset. Sau moi epoch, notebook luu `checkpoint_last.pt` vao `OUT_PUT_DIR`; file nay duoc ghi de bang trang thai moi nhat de tranh mat model khi Colab/runtime dung giua chung. Khi training ket thuc binh thuong, notebook van luu them checkpoint cuoi `final_no_context_dinov3_vits16_cars.pt`.


In [ ]:
def train_one_epoch(network, loader):
    network.train()
    total_loss = 0.0
    all_targets = []
    all_predictions = []

    progress = tqdm(loader, desc='train', leave=False)
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = network(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)
        all_targets.extend(labels.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    return {
        'loss': total_loss / len(loader.dataset),
        'accuracy': float(np.mean(np.asarray(all_targets) == np.asarray(all_predictions))),
    }


def build_checkpoint_payload(epoch, history):
    return {
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': list(history),
        'class_names': class_names,
        'architecture': BACKBONE_NAME,
        'feature_dim': FEATURE_DIM,
        'dinov3_model_name': DINOV3_MODEL_NAME,
        'dinov3_repo_dir': str(DINOV3_REPO_DIR),
        'dinov3_weights': str(resolve_dinov3_weights()),
        'dataset': 'Stanford Cars',
        'dataset_root': str(DATASET_ROOT),
        'standard_train_size': len(train_dataset),
        'standard_test_size': len(test_dataset),
        'official_test_has_labels': bool('class_id' in test_annotations.columns),
        'image_size': IMAGE_SIZE,
        'mean': imagenet_mean,
        'std': imagenet_std,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'use_bbox_crop': USE_BBOX_CROP,
        'pretraining': 'DINOv3',
    }


# Doi thanh True neu muon tiep tuc train tu OUT_PUT_DIR/checkpoint_last.pt.
RESUME_FROM_LAST_CHECKPOINT = False
history = []
start_epoch = 1
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    saved_checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(saved_checkpoint['model_state_dict'])

    if 'optimizer_state_dict' in saved_checkpoint:
        optimizer.load_state_dict(saved_checkpoint['optimizer_state_dict'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(DEVICE)

    if 'scheduler_state_dict' in saved_checkpoint:
        scheduler.load_state_dict(saved_checkpoint['scheduler_state_dict'])

    history = saved_checkpoint.get('history', [])
    start_epoch = int(saved_checkpoint.get('epoch', 0)) + 1
    print(f'Resumed from epoch {start_epoch - 1}: {last_checkpoint_path.resolve()}')

for epoch in range(start_epoch, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader)
    scheduler.step()

    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'learning_rate': scheduler.get_last_lr()[0],
    })

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"train top-1 accuracy {train_metrics['accuracy']:.4f} | "
        f"lr {scheduler.get_last_lr()[0]:.2e}"
    )

    torch.save(build_checkpoint_payload(epoch, history), last_checkpoint_path)
    print(f'Saved latest checkpoint: {last_checkpoint_path.resolve()}')

final_epoch = history[-1]['epoch'] if history else 0
torch.save(build_checkpoint_payload(final_epoch, history), checkpoint_path)
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
print('Saved final checkpoint:', checkpoint_path.resolve())
print('Saved latest checkpoint:', last_checkpoint_path.resolve())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df.plot(x='epoch', y='train_loss', ax=axes[0], marker='o', markevery=max(1, len(history_df) // 10), title='Training loss')
history_df.plot(x='epoch', y='train_accuracy', ax=axes[1], marker='o', markevery=max(1, len(history_df) // 10), title='Training top-1 accuracy')
plt.tight_layout()
plt.show()

## 7. Official test prediction

Standard test split trong `archive (10)` không có nhãn, nên không thể tính `test_top1_accuracy` hoặc confusion matrix trực tiếp. Cell này chạy model trên toàn bộ `cars_test/cars_test` và lưu file `official_test_predictions.csv` gồm tên ảnh, class dự đoán, tên dòng xe và xác suất.

In [ ]:
evaluation_checkpoint_path = checkpoint_path if checkpoint_path.is_file() else last_checkpoint_path
saved_checkpoint = torch.load(evaluation_checkpoint_path, map_location=DEVICE)
model.load_state_dict(saved_checkpoint['model_state_dict'])
model.eval()
print('Loaded checkpoint for prediction:', evaluation_checkpoint_path.resolve())

def predict_unlabeled_loader(network, loader, top_k=TOP_K):
    rows = []
    top_k = min(top_k, NUM_CLASSES)
    with torch.inference_mode():
        for images, file_names in tqdm(loader, leave=False):
            logits = network(images.to(DEVICE))
            probs = torch.softmax(logits, dim=1).cpu()
            values, indices = torch.topk(probs, k=top_k, dim=1)
            for fname, value_row, index_row in zip(file_names, values, indices):
                top_class_ids = [int(index) for index in index_row]
                rows.append({
                    'fname': fname,
                    'predicted_class_id': top_class_ids[0],
                    'predicted_car_name': class_names[top_class_ids[0]],
                    'confidence': float(value_row[0]),
                    'top_k_class_ids': '|'.join(str(index) for index in top_class_ids),
                    'top_k_car_names': '|'.join(class_names[index] for index in top_class_ids),
                    'top_k_probabilities': '|'.join(f'{float(value):.6f}' for value in value_row),
                })
    return pd.DataFrame(rows)

test_prediction_df = predict_unlabeled_loader(model, test_loader, top_k=TOP_K)
test_prediction_df.to_csv(OUTPUT_DIR / 'official_test_predictions.csv', index=False)
display(test_prediction_df.head())
print('Saved official test predictions:', (OUTPUT_DIR / 'official_test_predictions.csv').resolve())

with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump({
        'official_test_has_labels': bool('class_id' in test_annotations.columns),
        'official_test_metric_available': False,
        'note': 'cars_test_annos.mat trong archive (10) không có class label, nên notebook chỉ xuất prediction cho standard test split.',
        'num_classes': NUM_CLASSES,
        'standard_train_size': len(train_dataset),
        'standard_test_size': len(test_dataset),
        'architecture': BACKBONE_NAME,
        'pretraining': 'DINOv3',
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'seed': SEED,
        'use_bbox_crop': USE_BBOX_CROP,
    }, file, ensure_ascii=False, indent=2)


## 8. Dự đoán tên xe và class_id cho một ảnh

Hàm inference trả về `class_id` trong khoảng `0..195`, tên dòng xe tương ứng và xác suất. Ví dụ `PREDICT_IMAGE` mặc định là ảnh `00001.jpg` trong `cars_train/cars_train`.

In [ ]:
def predict_one_image(image_path, network=model, top_k=TOP_K):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy ảnh: {image_path.resolve()}')
    image = Image.open(image_path).convert('RGB')
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)
    network.eval()
    with torch.no_grad():
        probabilities = torch.softmax(network(tensor), dim=1)[0]
    top_k = min(top_k, NUM_CLASSES)
    values, indices = torch.topk(probabilities, k=top_k)
    result = pd.DataFrame({
        'class_id': [int(index) for index in indices.cpu()],
        'car_name': [class_names[int(index)] for index in indices.cpu()],
        'probability': values.cpu().numpy(),
    })
    display(image)
    print(f"Predicted class_id: {result.iloc[0]['class_id']}")
    print(f"Predicted car name: {result.iloc[0]['car_name']}")
    display(result.style.format({'probability': '{:.4%}'}))
    return result

# Sửa PREDICT_IMAGE nếu muốn dự đoán ảnh khác.
prediction = predict_one_image(PREDICT_IMAGE)

## 9. Tải checkpoint và dự đoán ở phiên chạy khác

Checkpoint `outputs/baseline_no_tacs_cars/final_no_context_dinov3_vits16_cars.pt` lưu weights của wrapper DINOv3 ViT-S/16, class names, preprocessing metadata và cấu hình Stanford Cars. Khi load lại cần DINOv3 repo/checkpoint tương ứng.

Neu training bi dung giua chung, ban co the load `checkpoint_last.pt` trong `OUT_PUT_DIR` hoac dat `RESUME_FROM_LAST_CHECKPOINT = True` o cell training de tiep tuc tu epoch moi nhat.


In [ ]:
def load_baseline_for_inference(checkpoint_file, device=DEVICE):
    saved = torch.load(checkpoint_file, map_location=device)
    weights = saved.get('dinov3_weights', resolve_dinov3_weights())
    loaded_model = DinoV3NoContextClassifier(
        repo_dir=saved.get('dinov3_repo_dir', DINOV3_REPO_DIR),
        model_name=saved.get('dinov3_model_name', saved.get('architecture', DINOV3_MODEL_NAME)),
        weights=weights,
        num_classes=len(saved['class_names']),
        feature_dim=saved.get('feature_dim', FEATURE_DIM),
    )
    loaded_model.load_state_dict(saved['model_state_dict'])
    loaded_model.to(device).eval()
    return loaded_model, saved['class_names']

print('Checkpoint:', checkpoint_path.resolve())
print('Class IDs: 0..195')
print('Pretraining:', 'DINOv3')

## 10. Phạm vi và lưu ý reproduction

- Đây là No-Context baseline: chỉ một ảnh đầu vào, không candidate pool và không context image.
- Dataset dùng đúng standard split của Stanford Cars trong `archive (10)`: 8,144 ảnh train có nhãn và 8,041 ảnh official test.
- Không tạo validation split từ train.
- Do `cars_test_annos.mat` trong bản này không có `class`, notebook không tính được official test accuracy; thay vào đó lưu `official_test_predictions.csv`.
- Nếu bạn có file test labels riêng, có thể thêm trường `class_id` vào `test_annotations` rồi viết thêm cell tính official test accuracy trên `test_loader` có nhãn.
- Model: `dinov3_vits16`, checkpoint DINOv3 ViT-S/16 LVD-1689M, ảnh 256x256, batch 64, 100 epochs, AdamW, cosine annealing.
- Có thể bật `USE_BBOX_CROP=True` để crop theo bounding box xe trước khi transform nếu muốn thử cải thiện chất lượng ảnh đầu vào.